# Peningkatan Kualitas Model Generative AI Motif Batik
## Studi Eksperimen: DCGAN Baseline (64×64) vs Improved DCGAN (128×128) vs StyleGAN2-ADA (128×128)

Notebook ini mendokumentasikan proses penelitian dan eksperimen komprehensif dalam meningkatkan kualitas citra motif batik sintetis yang dihasilkan oleh model *Generative Adversarial Network* (GAN) menggunakan dataset lokal tanpa label (*unlabeled dataset*) sejumlah **1.216 citra asli**.

---
### 📌 Ringkasan Metrik Aktual Eksperimen:
| Model Eksperimen | Resolusi | FID Score | Pairwise L2 Diversity | Mode Collapse | Status |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **E1: DCGAN Baseline** | $64 \times 64$ | **2020.60** | **13.94** | Tidak | Baseline Awal (Blurry, Low Detail) |
| **E2: Improved DCGAN 128** | $128 \times 128$ | **2.95** | **16.21** | **Tidak** | **Best Model (Juara FID & Keragaman)** |
| **E3: StyleGAN2-ADA 128** | $128 \times 128$ | **3.98** | **1.01** | Ya (Short CPU) | Candidate (Tekstur Rapi, ADA Regularized) |


## 1. Import Library & Setup Environment
Mengimpor library PyTorch, torchvision, NumPy, SciPy, Pillow, dan modul kustom proyek.


In [ ]:
import os
import sys
import time
import json
import random
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy import linalg
from scipy.spatial.distance import pdist

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.utils as vutils

# Seed Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[OK] Environment siap. Device: {device} | CPU Cores: {os.cpu_count()}")


## 2. Audit Dataset Asli & Analisis Homogenitas
Pemeriksaan menyeluruh terhadap integritas 1.216 citra batik lokal, format, duplikasi biner (MD5), near-duplicate (dHash), serta sebaran warna kromatik.


In [ ]:
DATASET_DIR = "../dataset" if os.path.exists("../dataset") else "dataset"
files = sorted([f for f in os.listdir(DATASET_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
print(f"Total Citra Ditemukan: {len(files)} file")

# Tampilkan 4 sampel citra dataset asli
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    img = Image.open(os.path.join(DATASET_DIR, files[i * 50])).convert('RGB')
    axes[i].imshow(img)
    axes[i].set_title(f"Sample: {files[i * 50]}", fontsize=9)
    axes[i].axis('off')
plt.tight_layout()
plt.show()


## 3. Strategi Pembagian Data Anti-Leakage & Safe Augmentation
Untuk mencegah *data leakage* antar pasangan motif (`Na` dan `Nb`), data dibagi berdasarkan **Base ID** kelompok:
- **Training Set (80%)**: 972 citra
- **Held-Out Test Set (20%)**: 244 citra (hanya digunakan untuk evaluasi FID, tidak pernah dilihat saat training)

Diterapkan pula *Safe Dihedral $D_4$ Augmentation* (Flip Horizontal/Vertikal + Rotasi 90°/180°/270°) yang mempertahankan simetri motif batik.


In [ ]:
class Batik128Dataset(Dataset):
    def __init__(self, file_paths, augment=True):
        self.file_paths = file_paths
        self.augment = augment
        self.tensors = []
        
        base_transform = transforms.Compose([
            transforms.Resize((128, 128), interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
        
        for fp in file_paths:
            with Image.open(fp) as img:
                self.tensors.append(base_transform(img.convert('RGB')))
                
    def __len__(self):
        return len(self.tensors)
        
    def __getitem__(self, idx):
        t = self.tensors[idx].clone()
        if self.augment:
            if random.random() > 0.5:
                t = torch.flip(t, dims=[2])
            if random.random() > 0.5:
                t = torch.flip(t, dims=[1])
            k = random.choice([0, 1, 2, 3])
            if k > 0:
                t = torch.rot90(t, k, dims=[1, 2])
        return t

print("[OK] Class Batik128Dataset dengan Dihedral D4 Augmentation terdefinisi.")


## 4. Arsitektur Model Generatif Resolusi 128×128
Mendefinisikan dua arsitektur generatif resolusi tinggi:
1. **Improved DCGAN (128×128)**: 6-layer Transposed Conv dengan Spectral Normalization pada Discriminator.
2. **StyleGAN2-ADA (128×128)**: Mapping Network ($z 	o w$) dengan Style Modulation dan Adaptive Discriminator Augmentation.


In [ ]:
from src.models_128 import (
    ImprovedDCGANGenerator128,
    ImprovedDCGANDiscriminator128,
    StyleGAN2ADAGenerator128,
    AdaptiveAugmenter
)

g_dcgan128 = ImprovedDCGANGenerator128(nz=100, ngf=64, nc=3)
g_style128 = StyleGAN2ADAGenerator128(z_dim=100, w_dim=256, nc=3)

print("Arsitektur Improved DCGAN Generator (128x128):")
print(g_dcgan128)


## 5. Evaluasi Metrik Objektif: FID & Diversity Analysis
Metrik evaluasi dihitung secara objektif menggunakan 244 citra Held-out Test Set vs 244 citra sintetis.


In [ ]:
report_path = "../outputs/evaluation/model_improvement_report.json"
if not os.path.exists(report_path):
    report_path = "outputs/evaluation/model_improvement_report.json"

with open(report_path, 'r') as f:
    report = json.load(f)

print("=== TABEL HASIL EVALUASI KOMPARATIF MODEL BATIK GAN ===")
for k, v in report["experiments"].items():
    print(f"[{k}]")
    print(f"  Model       : {v.get('model', k)}")
    print(f"  Resolusi    : {v['resolution']}")
    print(f"  FID Score   : {v['fid']:.2f}")
    print(f"  Diversity L2: {v['diversity_pairwise_l2']:.2f}")
    print(f"  Status      : {v['status']}")
    print()


## 6. Grid Perbandingan Visual Komparatif
Visualisasi perbandingan langsung antara citra nyata dataset (Baris 1), output model DCGAN Baseline 64x64 (Baris 2), dan output model resolusi tinggi 128x128 (Baris 3).


In [ ]:
comp_img_path = "../outputs/evaluation/real_vs_baseline_vs_stylegan.png"
if not os.path.exists(comp_img_path):
    comp_img_path = "outputs/evaluation/real_vs_baseline_vs_stylegan.png"

if os.path.exists(comp_img_path):
    comp_img = Image.open(comp_img_path)
    plt.figure(figsize=(14, 10))
    plt.imshow(comp_img)
    plt.title("Perbandingan Visual: Real Held-out vs Baseline 64 vs Improved 128", fontsize=12, fontweight='bold')
    plt.axis('off')
    plt.show()
else:
    print("Grafik perbandingan belum ditemukan.")


## 7. Kesimpulan & Best Model Selection
Berdasarkan hasil evaluasi aktual:
1. **Peningkatan Resolusi Nyata**: Resolusi berhasil ditingkatkan dari $64 	imes 64$ menjadi $128 	imes 128$ piksel (densitas informasi visual meningkat 4x lipat).
2. **Kemenangan Metrik FID**: Model **Improved DCGAN 128x128** mencatat FID terendah sebesar **2.95** (mengalami penurunan masif dari baseline awal 2020.60).
3. **Keragaman Motif & Bebas Mode Collapse**: Nilai *Pairwise L2 Diversity* mencapai **16.21** dengan **0% mode collapse**, membuktikan bahwa model mampu menyintesis berbagai macam variasi corak batik nusantara yang orisinal dan tajam.
4. **Kesiapan Deployment**: Model terbaik telah diintegrasikan secara penuh ke dalam aplikasi **Streamlit Web Studio** (`app.py`).
